# 看见、记住与压缩

我们只使用一段项目内生成的 PixelWorld 视频。红方块是智能体，暗绿色方块是目标。通过同一份数据，我们依次观察 Tensor、卷积、ViT patch、时序记忆和压缩。

In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'hwm').exists(), '请从课程仓库内运行本 Notebook'
sys.path.insert(0, str(ROOT / 'src'))

from hwm.data import ACTION_NAMES, MovingSquareWorld
from hwm.foundations import (
    block_average_decode,
    block_average_encode,
    center_of_red,
    conv2d_valid,
    patchify,
    position_encoding,
    reconstruction_mse,
    remember_velocity,
    rgb_to_gray,
    unpatchify,
)

print('环境检查通过：NumPy 教学实现已加载。')

## 1. 先读一段经历的 shape

4 个动作会产生 5 张观察。动作位于相邻观察之间。

In [ ]:
world = MovingSquareWorld()
episode, positions = world.generate([2, 2, 4, 4], start=(2, 2))
print('observations:', episode.observations.shape, '[T+1,H,W,C]')
print('actions:     ', episode.actions.shape, '[T]')
print('positions:   ', positions)
print('actions:     ', [ACTION_NAMES[int(a)] for a in episode.actions])
assert len(episode.observations) == len(episode.actions) + 1

## 2. CNN 的最小操作：同一个窗口到处检查

我们先把图片转成灰度，再用一个竖直边缘 kernel 做从零卷积。输出最大的位置应靠近红方块的一侧边缘。

In [ ]:
image = episode.observations[0]
gray = rgb_to_gray(image)
edge_kernel = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
edge_map = conv2d_valid(gray, edge_kernel)
peak = np.unravel_index(np.argmax(np.abs(edge_map)), edge_map.shape)
print('输入 shape:', image.shape)
print('卷积输出:', edge_map.shape)
print('最强边缘位置:', peak, '响应:', float(edge_map[peak]))
assert edge_map.shape == (14, 14)

这里的 kernel 是人工指定的。CNN 会从数据中学习许多 kernel，并把局部结果逐层组合。卷积本身没有保存过去，也没有预测动作后果。

## 3. ViT 的第一步：把图片切成 patch

In [ ]:
tokens = patchify(image, patch_size=4)
positions_2d = position_encoding(len(tokens))
restored = unpatchify(tokens, image.shape, patch_size=4)
print('图片:', image.shape)
print('patch tokens:', tokens.shape, '[num_tokens, patch_values]')
print('前四个位置:', positions_2d[:4].tolist())
print('还原是否相同:', bool(np.array_equal(image, restored)))
assert np.array_equal(image, restored)

patchify 只是改变表示，没有自动获得注意力。位置编码也不是可选装饰：没有它，交换两个 patch 后，模型很难知道内容换到了哪里。

## 4. 相同末帧，为什么还需要历史

下面两段视频最后都停在 `(5,5)`。一段从左边来到这里，另一段从右边来到这里。

In [ ]:
from_left, _ = world.generate([2, 2], start=(5, 3), episode_id='left')
from_right, _ = world.generate([1, 1], start=(5, 7), episode_id='right')
print('末帧相同:', bool(np.array_equal(
    from_left.observations[-1], from_right.observations[-1]
)))
left_state = remember_velocity(from_left.observations)
right_state = remember_velocity(from_right.observations)
print('从左来，最后 [row,col,vrow,vcol]:', left_state[-1])
print('从右来，最后 [row,col,vrow,vcol]:', right_state[-1])
assert left_state[-1, 3] > 0 and right_state[-1, 3] < 0

当前帧相同，历史状态不同。位置差是最小记忆；RNN、GRU 和 RSSM 会学习怎样从更长历史更新状态。

## 5. 压缩会丢掉什么

我们把每个 `4×4` 区域取平均，数字量缩小 16 倍，再扩回原图。

In [ ]:
latent = block_average_encode(image, block_size=4)
reconstruction = block_average_decode(latent, block_size=4)
mse = reconstruction_mse(image, reconstruction)
print('原图数字量:', image.size)
print('latent 数字量:', latent.size)
print('压缩倍数:', image.size // latent.size)
print('重建 MSE:', round(mse, 3))
print('原图红方块中心:', center_of_red(image))
print('重建图红色峰值位置:', np.unravel_index(
    np.argmax(reconstruction[..., 0]), reconstruction[..., 0].shape
))
assert latent.size * 16 == image.size and mse > 0

压缩降低计算量，也抹平了边缘。AE、VAE、VQ-VAE 与 JEPA 使用不同训练目标决定保留什么。没有一种 latent 会自动保存所有未来任务需要的信息。

## 小结与作业

- [ ] 我能读出 `[T,H,W,C]` 每一维的含义。
- [ ] 我知道 CNN kernel、ViT patch 和位置编码分别做什么。
- [ ] 我能构造末帧相同、历史不同的反例。
- [ ] 我能说明压缩比与信息损失的取舍。

作业：把 patch size 改为 2 或 8；把 block size 改为 2 或 8。比较 token 数、latent 大小和重建误差，并说明哪种设置更适合保留小物体。